##Recall Postcell Class

In [ ]:
#!pip install postcell

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/2026/Markoding/Week-1-pi-integrasi-ai.git #silahkan ganti dengan lokasi hasil clone kalian

!git status

In [ ]:
%reload_ext postcell
%postcell register

# Hands-on: AI Recommendation System dengan Pretrained Model
**Studi kasus:** rekomendasi produk ramah lingkungan berdasarkan preferensi pengguna  
**Platform:** Google Colab | **Level:** Pemula | **Estimasi:** 45–60 menit

## Tujuan pembelajaran
Setelah menyelesaikan notebook ini, peserta dapat:
1. Menjelaskan perbedaan pretrained model dan training dari nol.
2. Mengubah teks menjadi *embedding* menggunakan Sentence Transformers.
3. Menghitung kemiripan teks dengan cosine similarity.
4. Membuat fungsi rekomendasi Top-K.
5. Menguji sistem dan mendiskusikan keterbatasannya.

> **Catatan pengajar:** Tekankan bahwa model bahasa tidak otomatis menjadi sistem rekomendasi. Model menghasilkan embedding teks; logika similarity dan ranking yang kita tulis mengubah embedding itu menjadi rekomendasi.

## 1. Gambaran alur sistem

1. **Input pengguna** — misalnya, “Saya ingin mengurangi penggunaan plastik.”
2. **Katalog produk** — setiap item memiliki nama dan deskripsi.
3. **Pretrained embedding model** — mengubah teks menjadi vektor numerik.
4. **Cosine similarity** — membandingkan vektor preferensi dengan vektor produk.
5. **Ranking** — menampilkan produk dengan skor tertinggi.

**Istilah penting**
- **Pretrained model:** model yang bobotnya sudah dipelajari sebelumnya dari data besar.
- **Embedding:** representasi numerik dari teks.
- **Cosine similarity:** ukuran kemiripan arah dua vektor; nilai lebih tinggi berarti lebih mirip menurut representasi model.
- **Top-K:** sejumlah K rekomendasi teratas.

**Batasan:** contoh ini adalah *content-based recommendation*, bukan collaborative filtering. Sistem tidak mempelajari pola pembelian atau perilaku pengguna lain.

## 2. Instalasi library
Jalankan sel berikut. Google Colab biasanya sudah menyediakan Python; kita hanya perlu memasang library tambahan.

In [1]:
%pip -q install sentence-transformers scikit-learn pandas

## 3. Import library
Kita memakai `pandas` untuk katalog, `SentenceTransformer` untuk embedding, dan `cosine_similarity` untuk menghitung kemiripan.

In [2]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## 4. Membuat dataset mini
Dataset ini dibuat langsung di notebook agar peserta bisa fokus pada konsep, tanpa perlu mengunduh file eksternal. Dalam proyek nyata, katalog bisa berasal dari CSV, database, atau API.

In [3]:
products = pd.DataFrame([
    {
        "nama": "Botol minum reusable",
        "kategori": "Perlengkapan",
        "deskripsi": "Botol minum yang dapat digunakan berulang kali untuk mengurangi sampah plastik"
    },
    {
        "nama": "Tas belanja kain",
        "kategori": "Perlengkapan",
        "deskripsi": "Tas kain yang dapat digunakan kembali saat berbelanja tanpa kantong plastik"
    },
    {
        "nama": "Sedotan bambu",
        "kategori": "Perlengkapan",
        "deskripsi": "Sedotan ramah lingkungan yang bisa digunakan kembali"
    },
    {
        "nama": "Lampu tenaga surya",
        "kategori": "Energi",
        "deskripsi": "Lampu hemat energi yang menggunakan tenaga matahari"
    },
    {
        "nama": "Komposter rumah tangga",
        "kategori": "Pengelolaan sampah",
        "deskripsi": "Alat untuk mengolah sampah organik menjadi kompos"
    },
    {
        "nama": "Kotak makan reusable",
        "kategori": "Perlengkapan",
        "deskripsi": "Wadah makanan yang dapat digunakan kembali untuk mengurangi sampah kemasan"
    }
])

products

,nama,kategori,deskripsi
0,Botol minum reusable,Perlengkapan,Botol minum yang dapat digunakan berulang kali...
1,Tas belanja kain,Perlengkapan,Tas kain yang dapat digunakan kembali saat ber...
2,Sedotan bambu,Perlengkapan,Sedotan ramah lingkungan yang bisa digunakan k...
3,Lampu tenaga surya,Energi,Lampu hemat energi yang menggunakan tenaga mat...
4,Komposter rumah tangga,Pengelolaan sampah,Alat untuk mengolah sampah organik menjadi kompos
5,Kotak makan reusable,Perlengkapan,Wadah makanan yang dapat digunakan kembali unt...


### Catatan diskusi
- Data apa saja yang dibutuhkan agar rekomendasi lebih baik?
- Apakah deskripsi produk yang terlalu pendek akan memengaruhi hasil?
- Apakah katalog ini cukup mewakili semua produk yang mungkin dibutuhkan pengguna?

Dataset kecil ini hanya untuk pembelajaran; hasilnya belum dapat dianggap sebagai evaluasi sistem produksi.

## 5. Memuat pretrained model
Kita menggunakan `sentence-transformers/all-MiniLM-L6-v2`, model embedding yang relatif ringan dan menghasilkan vektor 384 dimensi.

Model ini umumnya bekerja baik untuk teks bahasa Inggris. Contoh kita menggunakan bahasa Indonesia demi kemudahan kelas, sehingga kualitas semantic matching perlu diuji. Untuk aplikasi Indonesia yang serius, bandingkan juga model multilingual.

In [4]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)
print("Model berhasil dimuat:", MODEL_NAME)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model berhasil dimuat: sentence-transformers/all-MiniLM-L6-v2


## 6. Mengubah deskripsi produk menjadi embedding
Kita menghitung embedding produk satu kali, lalu menggunakannya berulang kali saat pengguna mengirim preferensi.

`normalize_embeddings=True` menormalkan vektor. Ini memudahkan perbandingan cosine similarity.

In [5]:
product_embeddings = model.encode(
    products["deskripsi"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Bentuk matriks embedding:", product_embeddings.shape)
print("Jumlah produk:", len(products))
print("Dimensi embedding per produk:", product_embeddings.shape[1])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Bentuk matriks embedding: (6, 384)
Jumlah produk: 6
Dimensi embedding per produk: 384


**Cara membaca output:** jika bentuknya `(6, 384)`, artinya ada 6 produk dan setiap produk direpresentasikan oleh 384 angka. Angka-angka ini bukan label produk; mereka merepresentasikan pola semantik yang dipelajari model.

## 7. Membuat fungsi rekomendasi
Fungsi berikut menerima preferensi dalam bentuk teks, membuat embedding untuk preferensi tersebut, menghitung similarity terhadap semua produk, lalu mengembalikan Top-K produk dengan skor tertinggi.

In [6]:
def recommend_products(preference, top_k=3):
    # Validasi input sederhana
    if not isinstance(preference, str) or not preference.strip():
        raise ValueError("Preferensi harus berupa teks yang tidak kosong.")

    if top_k < 1:
        raise ValueError("top_k harus minimal 1.")

    # Embedding preferensi pengguna
    query_embedding = model.encode(
        [preference],
        normalize_embeddings=True
    )

    # Skor kemiripan terhadap seluruh produk
    scores = cosine_similarity(
        query_embedding,
        product_embeddings
    )[0]

    # Ambil indeks dengan skor tertinggi
    ranked_indices = scores.argsort()[::-1][:min(top_k, len(products))]

    results = products.iloc[ranked_indices].copy()
    results["skor_kemiripan"] = scores[ranked_indices].round(3)

    return results.reset_index(drop=True)

### Memahami kode
- `model.encode([preference])`: mengubah input pengguna menjadi embedding.
- `cosine_similarity(...)`: menghasilkan skor untuk setiap produk.
- `argsort()[::-1]`: mengurutkan indeks dari skor tertinggi ke terendah.
- `top_k`: membatasi jumlah rekomendasi yang ditampilkan.

**Penting:** skor kemiripan bukan probabilitas dan bukan akurasi. Skor 0.70 tidak berarti peluang pengguna menyukai produk adalah 70%.

## 8. Uji rekomendasi pertama
Jalankan dengan preferensi pengguna berikut.

In [7]:
user_preference = (
    "Saya ingin mengurangi penggunaan plastik "
    "ketika membawa minuman dan makanan."
)

recommend_products(user_preference, top_k=3)

,nama,kategori,deskripsi,skor_kemiripan
0,Kotak makan reusable,Perlengkapan,Wadah makanan yang dapat digunakan kembali unt...,0.737
1,Botol minum reusable,Perlengkapan,Botol minum yang dapat digunakan berulang kali...,0.654
2,Tas belanja kain,Perlengkapan,Tas kain yang dapat digunakan kembali saat ber...,0.637


## 9. Eksperimen: ubah preferensi pengguna
Coba beberapa masukan berikut, lalu bandingkan urutan hasil rekomendasinya.

In [8]:
test_preferences = [
    "Saya ingin mengurangi sampah plastik saat berbelanja.",
    "Saya ingin menghemat energi di rumah.",
    "Saya ingin mengolah sampah makanan menjadi sesuatu yang berguna.",
    "Saya membutuhkan barang yang bisa dipakai berulang kali."
]

for preference in test_preferences:
    print("\nPreferensi:", preference)
    display(recommend_products(preference, top_k=3))


Preferensi: Saya ingin mengurangi sampah plastik saat berbelanja.


,nama,kategori,deskripsi,skor_kemiripan
0,Komposter rumah tangga,Pengelolaan sampah,Alat untuk mengolah sampah organik menjadi kompos,0.720
1,Tas belanja kain,Perlengkapan,Tas kain yang dapat digunakan kembali saat ber...,0.697
2,Botol minum reusable,Perlengkapan,Botol minum yang dapat digunakan berulang kali...,0.695



Preferensi: Saya ingin menghemat energi di rumah.


,nama,kategori,deskripsi,skor_kemiripan
0,Lampu tenaga surya,Energi,Lampu hemat energi yang menggunakan tenaga mat...,0.559
1,Komposter rumah tangga,Pengelolaan sampah,Alat untuk mengolah sampah organik menjadi kompos,0.454
2,Botol minum reusable,Perlengkapan,Botol minum yang dapat digunakan berulang kali...,0.444



Preferensi: Saya ingin mengolah sampah makanan menjadi sesuatu yang berguna.


,nama,kategori,deskripsi,skor_kemiripan
0,Kotak makan reusable,Perlengkapan,Wadah makanan yang dapat digunakan kembali unt...,0.703
1,Lampu tenaga surya,Energi,Lampu hemat energi yang menggunakan tenaga mat...,0.689
2,Komposter rumah tangga,Pengelolaan sampah,Alat untuk mengolah sampah organik menjadi kompos,0.681



Preferensi: Saya membutuhkan barang yang bisa dipakai berulang kali.


,nama,kategori,deskripsi,skor_kemiripan
0,Tas belanja kain,Perlengkapan,Tas kain yang dapat digunakan kembali saat ber...,0.662
1,Botol minum reusable,Perlengkapan,Botol minum yang dapat digunakan berulang kali...,0.646
2,Kotak makan reusable,Perlengkapan,Wadah makanan yang dapat digunakan kembali unt...,0.603


### Pertanyaan refleksi
1. Apakah perubahan kalimat mengubah urutan rekomendasi?
2. Apakah hasilnya masuk akal jika dilihat dari deskripsi produk?
3. Apakah ada rekomendasi yang terasa tidak relevan? Mengapa?
4. Bagaimana cara menguji kualitas sistem dengan lebih objektif?

**Saran pengajar:** minta peserta menciptakan dua preferensi sendiri dan menjelaskan mengapa produk tertentu muncul di posisi pertama.

## 10. Tampilkan semua skor
Untuk memahami ranking, kita bisa melihat skor semua produk, bukan hanya Top-K.

In [9]:
def score_all_products(preference):
    query_embedding = model.encode(
        [preference],
        normalize_embeddings=True
    )
    scores = cosine_similarity(query_embedding, product_embeddings)[0]

    result = products.copy()
    result["skor_kemiripan"] = scores.round(3)
    return result.sort_values("skor_kemiripan", ascending=False).reset_index(drop=True)

score_all_products("Saya ingin mengurangi sampah plastik.")

,nama,kategori,deskripsi,skor_kemiripan
0,Komposter rumah tangga,Pengelolaan sampah,Alat untuk mengolah sampah organik menjadi kompos,0.723
1,Botol minum reusable,Perlengkapan,Botol minum yang dapat digunakan berulang kali...,0.687
2,Kotak makan reusable,Perlengkapan,Wadah makanan yang dapat digunakan kembali unt...,0.640
3,Lampu tenaga surya,Energi,Lampu hemat energi yang menggunakan tenaga mat...,0.614
4,Tas belanja kain,Perlengkapan,Tas kain yang dapat digunakan kembali saat ber...,0.606
5,Sedotan bambu,Perlengkapan,Sedotan ramah lingkungan yang bisa digunakan k...,0.522


## 11. Tantangan untuk peserta
Kerjakan secara mandiri atau berpasangan.

**Tantangan A — Tambah item**
- Tambahkan minimal 3 produk baru.
- Berikan deskripsi yang spesifik.
- Jalankan ulang sel dataset dan embedding produk sebelum menguji rekomendasi.

**Tantangan B — Ubah Top-K**
- Bandingkan `top_k=1`, `top_k=3`, dan `top_k=5`.
- Apa yang berubah pada jumlah hasil dan relevansinya?

**Tantangan C — Jelaskan keterbatasan**
- Bagaimana jika pengguna menulis dalam bahasa Indonesia tetapi model kurang memahami nuansanya?
- Bagaimana jika katalog tidak memiliki item yang sesuai?
- Apakah skor similarity tinggi selalu berarti pengguna akan menyukai produk?

**Tantangan D — Perbaikan**
Usulkan cara meningkatkan sistem, misalnya menambah metadata, filter kategori, evaluasi dengan contoh berlabel, atau mencoba model multilingual.

## 12. Troubleshooting
- **Model lama dimuat:** unduhan model pertama kali dapat memerlukan waktu dan koneksi internet.
- **RAM atau runtime penuh:** gunakan dataset lebih kecil atau restart runtime, lalu jalankan sel dari atas.
- **Mengubah dataset tetapi hasil tidak berubah:** hitung ulang `product_embeddings` setelah dataset diubah.
- **Rekomendasi bahasa Indonesia kurang relevan:** model ini berorientasi kuat pada bahasa Inggris; uji model multilingual dan bandingkan hasil pada contoh yang sama.
- **`NameError`:** pastikan sel yang mendefinisikan variabel/fungsi sudah dijalankan.

Jika hasil kurang tepat, itu tidak otomatis berarti kode salah. Model, bahasa, kualitas deskripsi, dan isi katalog turut menentukan hasil.

## 13. Ringkasan
Dalam hands-on ini kita membangun sistem rekomendasi sederhana dengan alur:

**Teks preferensi → embedding pretrained → cosine similarity → ranking → Top-K rekomendasi**

Kita tidak melakukan fine-tuning atau training model dari nol. Kita menggunakan model pretrained sebagai pembuat embedding, kemudian membuat logika rekomendasi di atas embedding tersebut.

### Pengembangan setelah kelas
1. Membaca katalog dari CSV atau Supabase.
2. Membuat API rekomendasi menggunakan FastAPI.
3. Membuat antarmuka web dengan Streamlit atau Next.js.
4. Menguji model multilingual untuk teks bahasa Indonesia.
5. Mengevaluasi relevansi dengan penilaian pengguna dan metrik ranking yang sesuai.

**Referensi**
- Model card: https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
- Dokumentasi Sentence Transformers: https://www.sbert.net/
- Dokumentasi scikit-learn cosine similarity: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html